# Stage 2: Train Latent Diffusion (Unified IMM/DDPM)

This notebook trains the Latent Diffusion Model on the latents of the frozen Base Model.
Supports:
1. **Diffusion Types**: `ddpm`, `imm` (Iterative Moment Matching).
2. **Models**: `vae`, `nvae`.
3. **Datasets**: `CelebA`, `CIFAR10`, `FFHQ`.


In [ ]:
%load_ext autoreload
%autoreload 2

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms, datasets
import numpy as np
import matplotlib.pyplot as plt
import os

from training.train_utils import set_seed, Model, VAE, CelebA
from training.nvae_model import NVAE
from training.datasets import FlatImageFolder
from training.imm_utils import ResidualDenoiser, IMM, OTFMSchedule, OtFmEulerFmParameterization
from training.ddpm_utils import DDPM

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
def get_dataloader(dataset_name, batch_size=128, image_size=32):
    if dataset_name == 'CelebA':
        transform = transforms.Compose([
            transforms.Resize((image_size, image_size)),
            transforms.CenterCrop(image_size),
            transforms.ToTensor(),
            transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
        ])
        dset = CelebA("./data/CelebA", split='train', download=True, transform=transform)
    elif dataset_name == 'CIFAR10':
        transform = transforms.Compose([
            transforms.Resize((image_size, image_size)),
            transforms.ToTensor(),
            transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
        ])
        dset = datasets.CIFAR10(root='./data/CIFAR10', train=True, download=True, transform=transform)
    elif dataset_name == 'FFHQ':
        transform = transforms.Compose([
            transforms.Resize((image_size, image_size)),
            transforms.CenterCrop(image_size),
            transforms.ToTensor(),
            transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
        ])
        dset = FlatImageFolder(root='./data/FFHQ', transform=transform)
    else:
        raise ValueError("Unknown dataset")
    return DataLoader(dset, batch_size=batch_size, shuffle=True, drop_last=True, num_workers=2, pin_memory=True)


In [ ]:
def train_latent_diffusion(diffusion_type, model_type, dataset_name, z_dim=64, n_epochs=50):
    save_path = f"checkpoints/best_{diffusion_type}_{model_type}_{dataset_name}.pth"
    base_ckpt = f"checkpoints/best_{model_type}_{dataset_name}.pth"
    
    if not os.path.exists(base_ckpt):
        print(f"Base model {base_ckpt} not found! Train Stage 1 first.")
        return
        
    IMG_RES = 32
    if model_type == 'vae':
        base_model = Model(3, 32, 3, 128, z_dim, 32, 32).to(device)
    else:
        base_model = NVAE(z_dim=z_dim, input_res=IMG_RES).to(device)
        
    base_model.load_state_dict(torch.load(base_ckpt, map_location=device))
    base_model.eval()
    for p in base_model.parameters(): p.requires_grad = False
    print(f"Loaded frozen {model_type} from {base_ckpt}")

    # Setup Diffusion
    if diffusion_type == 'imm':
        # IMM Config
        denoiser = ResidualDenoiser(
            input_dim=z_dim + 2, # x + noise_s + noise_t (1+1)
            model_dim=512,
            output_dim=z_dim,
            n_blocks=6
        )
        schedule = OTFMSchedule()
        param = OtFmEulerFmParameterization(sigma_data=1.0)
        diff_model = IMM(denoiser, param, D=z_dim, M=16).to(device)
    else: 
        # DDPM Config
        denoiser = ResidualDenoiser(
            input_dim=z_dim + 64,
            model_dim=512,
            output_dim=z_dim,
            n_blocks=6
        )
        diff_model = DDPM(denoiser, z_dim=z_dim, n_steps=1000, device=device).to(device)
        
    optimizer = torch.optim.Adam(diff_model.parameters(), lr=1e-4)
    dataloader = get_dataloader(dataset_name, batch_size=128, image_size=IMG_RES)
    best_loss = float('inf')

    print(f"Starting {diffusion_type.upper()} Training...")
    for epoch in range(n_epochs):
        diff_model.train()
        total_loss = 0
        steps = 0
        
        for x, _ in dataloader:
            x = x.to(device)
            
            with torch.no_grad():
                if model_type == 'vae':
                    mu, logvar = base_model.encoder(x)
                    z = base_model.reparameterize(mu, logvar)
                else:
                    mu_g, log_g, _, _, _, _ = base_model.encoder(x)
                    std_g = torch.exp(0.5 * log_g)
                    z = mu_g + torch.randn_like(mu_g) * std_g
            
            loss = diff_model.compute_loss(z)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            steps += 1
            
        avg_loss = total_loss / steps
        print(f"Epoch {epoch+1}: Loss {avg_loss:.5f}")
        
        if avg_loss < best_loss:
            best_loss = avg_loss
            torch.save(diff_model.state_dict(), save_path)

    print(f"Saved to {save_path}")
    return diff_model

In [ ]:
if __name__ == "__main__":
    set_seed(42)
    
    # Example Comparison Train
    # 1. Train DDPM
    train_latent_diffusion('ddpm', 'nvae', 'CIFAR10', n_epochs=10)
    
    # 2. Train IMM
    train_latent_diffusion('imm', 'nvae', 'CIFAR10', n_epochs=10)
    